# VibeVoice Mystery Narration - Google Colab

This notebook demonstrates how to use Microsoft's VibeVoice 1.5B model for generating high-quality mystery podcast narration on Google Colab's free GPU.

## What is VibeVoice?

- State-of-the-art TTS for long-form conversational audio
- No token limits (64K context = ~50,000 words)
- Perfect for mystery storytelling (dramatic pauses, suspenseful tone)
- Can clone voices from 5-15 second samples
- MIT License (commercial use allowed)

## Important: Text Format Requirement

**VibeVoice community fork requires `Speaker N:` format in your text!** 

### Correct Format

**Single-narrator mystery stories:**
```
Speaker 0: First paragraph of your story here.

Speaker 0: Second paragraph continues the tale.

Speaker 0: Third paragraph with more suspense.
```

**Multi-speaker dialogue:**
```
Speaker 0: The detective entered the room.
Speaker 1: Where were you last night?
Speaker 2: I was home alone.
Speaker 0: Your alibi doesn't check out.
```

### Format Rules

- Case-insensitive: `Speaker 0:`, `speaker 0:`, or `SPEAKER 0:` all work
- Space after "Speaker" required: `Speaker 0:` not `Speaker0:`
- Colon required: `Speaker 0:` not `Speaker 0`
- Numbers start from 0 (use 0, 1, 2, 3 for up to 4 speakers)

### INCORRECT Formats (cause errors)

```
[speaker 0] Your text...          ❌ Wrong - uses brackets
NARRATOR: Your text...            ❌ Wrong - wrong keyword
It was a dark night...            ❌ Wrong - no speaker label
```

**Without proper format, you'll get:**
- `"Could not parse line"` warnings
- `"No valid speaker lines found in script"` error

## Requirements

- Google Colab with GPU runtime (free tier works!)
- 12GB+ GPU VRAM (T4, A100, or V100)
- ~10 minutes for setup + generation

---

## Step 1: Check GPU Availability

First, verify you have a GPU with sufficient VRAM:

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✓ GPU detected: {gpu_name}")
    print(f"✓ VRAM: {gpu_memory:.1f} GB")
    
    if gpu_memory < 12:
        print(f"\n⚠️  WARNING: GPU has only {gpu_memory:.1f}GB VRAM.")
        print("   Minimum 12GB recommended. Generation may fail.")
    else:
        print("\n✓ GPU has sufficient VRAM for VibeVoice!")
else:
    print("❌ ERROR: No GPU detected.")
    print("   Go to Runtime > Change runtime type > GPU")
    raise RuntimeError("GPU required for VibeVoice")

## Step 2: Install VibeVoice

Install the community fork of VibeVoice (official repo is disabled):

In [ ]:
%%bash
# Install VibeVoice community fork
pip install -q git+https://github.com/vibevoice-community/VibeVoice.git

# Install audio processing dependencies
pip install -q soundfile librosa pydub

echo "✓ Installation complete!"

## Step 3: Prepare Your Mystery Story

Paste your mystery text here (can be very long - up to 50,000 words!):

In [ ]:
# Your mystery story text
# IMPORTANT: VibeVoice community fork requires "Speaker N: text" format!
# For single-narrator stories, use "Speaker 0:" for all paragraphs

MYSTERY_STORY = """Speaker 0: It was a fog-laden morning when I stumbled upon a peculiar bottle at a local estate sale in Philadelphia. Its emerald green hue caught the dim light, and embossed on its side were the words: SWAIM'S PANACEA PHILADA. The bottle's antiquity was evident, but it was the unsettling aura surrounding it that piqued my curiosity.

Speaker 0: This was no ordinary tonic. It was the creation of William Swaim, a Philadelphia businessman who became one of the most notorious patent medicine purveyors of the 19th century. Swaim claimed the title "Dr."—though he never earned a formal medical degree. In those days, styling oneself as a physician was common among patent medicine sellers, lending credibility to their remedies. Swaim's panacea promised cures for nearly every chronic ailment: scrofula, syphilis, ulcers, and skin diseases.

Speaker 0: The medicine's ingredients were potent—and dangerous. Its primary component, mercury dichloride, could cause serious illness with repeated use. To mask its bitter metallic taste, Swaim blended in sarsaparilla, licorice, and wintergreen. Though some patients reported dramatic improvements, others suffered serious or even fatal consequences.

Speaker 0: [PASTE YOUR FULL STORY HERE - Remember to prefix each paragraph with "Speaker 0: "]
""".strip()

# Show word count
word_count = len(MYSTERY_STORY.replace("Speaker 0:", "").split())
print(f"Story length: {word_count} words")
print(f"Estimated audio: ~{word_count / 150:.1f} minutes")
print(f"Estimated generation time: ~{word_count / 150 * 6:.0f} minutes (0.15x real-time)")
print(f"\n✓ Story formatted with 'Speaker 0:' labels for VibeVoice")

## Step 4: Upload Voice Reference (Optional)

You can either:
1. **Upload a male narrator voice sample** (5-15 seconds, MP3/WAV)
2. **Skip this step** to use VibeVoice's default voice

To upload, click the folder icon on the left sidebar, then upload your audio file.

In [ ]:
import librosa
from pathlib import Path

# Option 1: Use uploaded voice reference
VOICE_REFERENCE_PATH = None  # Set to "/content/your_voice.mp3" if uploaded

# Option 2: Or use Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# VOICE_REFERENCE_PATH = "/content/drive/MyDrive/narrator_voice.mp3"

voice_audio = None
if VOICE_REFERENCE_PATH and Path(VOICE_REFERENCE_PATH).exists():
    voice_audio, sr = librosa.load(VOICE_REFERENCE_PATH, sr=24000, mono=True)
    duration = len(voice_audio) / sr
    print(f"✓ Loaded voice reference: {duration:.1f} seconds")
    
    if duration < 3:
        print("⚠️  WARNING: Reference audio <3 seconds. 5-15 seconds recommended.")
    elif duration > 30:
        print("⚠️  Trimming to first 15 seconds...")
        voice_audio = voice_audio[:int(15 * sr)]
else:
    print("ℹ️  No voice reference provided. Using VibeVoice default voice.")

## Step 5: Load VibeVoice Model

This downloads ~5.4GB model weights (takes 1-2 minutes):

In [ ]:
from vibevoice.modular.modeling_vibevoice_inference import (
    VibeVoiceForConditionalGenerationInference
)
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor
import torch

print("Loading VibeVoice processor...")
processor = VibeVoiceProcessor.from_pretrained("microsoft/VibeVoice-1.5B")
print("✓ Processor loaded")

print("\nLoading VibeVoice-1.5B model (5.4GB download)...")
print("This may take 1-2 minutes...")
model = VibeVoiceForConditionalGenerationInference.from_pretrained(
    "microsoft/VibeVoice-1.5B",
    torch_dtype=torch.bfloat16
).to("cuda")
print("✓ Model loaded on GPU")

# Set inference steps (5=fast, 20=quality)
INFERENCE_STEPS = 5  # Change to 10-20 for better quality
model.set_ddpm_inference_steps(num_steps=INFERENCE_STEPS)
print(f"✓ Inference steps: {INFERENCE_STEPS}")

print("\n✓ VibeVoice ready for generation!")

## Step 6: Generate Mystery Narration

This generates the full audio. **Be patient** - it takes ~6 minutes per minute of audio:

In [ ]:
import torch
from IPython.display import Audio, display
import soundfile as sf
import re

# Configuration
CFG_SCALE = 1.3  # 1.0-2.0, higher = stricter text adherence

print("="*60)
print("GENERATING AUDIO")
print("="*60)

# Validate speaker format (must match: Speaker N: text)
speaker_pattern = r'^Speaker\s+(\d+)\s*:\s*(.*)$'
valid_lines = []
for line in MYSTERY_STORY.split('\n'):
    if line.strip():
        if re.match(speaker_pattern, line.strip(), re.IGNORECASE):
            valid_lines.append(line)

if not valid_lines:
    print("\n❌ ERROR: Story text must use 'Speaker N: text' format!")
    print("   Example: 'Speaker 0: Your story text here...'")
    print("   For single narrator, use 'Speaker 0:' for all paragraphs.")
    print("   For multiple speakers, use 'Speaker 0:', 'Speaker 1:', etc.\n")
    raise ValueError("Missing 'Speaker N:' labels in story text")

# Count speakers
speakers_found = set(re.findall(r'Speaker\s+(\d+)\s*:', MYSTERY_STORY, re.IGNORECASE))
print(f"✓ Found {len(speakers_found)} speaker(s): Speaker {', Speaker '.join(sorted(speakers_found))}")

# Prepare inputs
voice_samples_formatted = [[voice_audio]] if voice_audio is not None else None

inputs = processor(
    text=[MYSTERY_STORY],
    voice_samples=voice_samples_formatted,
    return_tensors="pt"
)

# Move to GPU (only move tensors, skip lists/other types)
inputs = {k: v.to("cuda") if torch.is_tensor(v) else v for k, v in inputs.items()}

# Estimate generation time
word_count = len(re.sub(r'Speaker\s+\d+\s*:', '', MYSTERY_STORY, flags=re.IGNORECASE).split())
estimated_audio_minutes = word_count / 150
estimated_gen_time = estimated_audio_minutes * 6

print(f"\nEstimated audio length: ~{estimated_audio_minutes:.1f} minutes")
print(f"Estimated generation time: ~{estimated_gen_time:.0f} minutes")
print(f"CFG scale: {CFG_SCALE}")
print(f"\nGenerating... (this will take a while)")
print("☕ Go get coffee! ☕\n")

# Generate
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        cfg_scale=CFG_SCALE,
        tokenizer=processor.tokenizer,
        generation_config={'do_sample': False},
        verbose=True  # Show progress
    )

# Extract audio - check if generation succeeded
if not hasattr(outputs, 'speech_outputs') or len(outputs.speech_outputs) == 0:
    print("\n❌ ERROR: No audio generated!")
    raise RuntimeError("Audio generation failed - no speech outputs")

audio_tensor = outputs.speech_outputs[0]

if audio_tensor.numel() == 0:
    print("\n❌ ERROR: Generated audio is empty!")
    raise RuntimeError("Generated audio is empty")

# Squeeze to remove batch dimension: [1, N] -> [N]
audio_array = audio_tensor.detach().cpu().float().numpy().ravel()
sample_rate = 24000

actual_duration = len(audio_array) / sample_rate / 60

print("\n" + "="*60)
print("✓ GENERATION COMPLETE!")
print("="*60)
print(f"✓ Audio duration: {actual_duration:.1f} minutes")
print(f"✓ Sample rate: {sample_rate} Hz")
print(f"✓ Audio samples: {len(audio_array):,}")

# Save audio
output_path = "/content/mystery_narration.wav"
sf.write(output_path, audio_array, sample_rate)
print(f"✓ Saved to: {output_path}")

# Play in notebook
print("\n🎧 Listen to your mystery narration:")
display(Audio(audio_array, rate=sample_rate))

## Step 6.5: Add Ending Snippet (Optional)

Add a professional ending to your mystery podcast! You can either:
1. **Upload your own ending.mp3** (music, credits, etc.)
2. **Skip this step** to use the narration as-is

**Configuration options:**
- **Silent gap**: Add pause between narration and ending (default: 1 second)
- **Crossfade**: Optional smooth transition (default: 0 = no overlap)

The ending will be appended with configurable gap and crossfade for professional results.

In [ ]:
from pathlib import Path

# Option 1: Upload your own ending.mp3 file
# Click the folder icon on the left, then upload your ending.mp3
ENDING_PATH = None  # Set to "/content/ending.mp3" if uploaded

# Option 2: Download the default ending from the repo
# Uncomment the lines below to use the default ending from the GitHub repo
# !wget -q https://raw.githubusercontent.com/linwang/antique_mystery_podcast/main/assets/ending.mp3 -O /content/ending.mp3
# ENDING_PATH = "/content/ending.mp3"

if ENDING_PATH and Path(ENDING_PATH).exists():
    print(f"✓ Ending file found: {ENDING_PATH}")
    ENDING_ENABLED = True
else:
    print("ℹ️  No ending file provided. Skipping ending snippet.")
    ENDING_ENABLED = False

In [ ]:
if ENDING_ENABLED:
    # Install ffmpeg (required for pydub to handle MP3)
    print("Installing ffmpeg...")
    !apt-get update -qq && apt-get install -y -qq ffmpeg > /dev/null 2>&1
    print("✓ ffmpeg installed")
    
    # Install pydub if not already installed
    try:
        from pydub import AudioSegment
    except ImportError:
        !pip install -q pydub
        from pydub import AudioSegment
    
    print("="*60)
    print("COMBINING AUDIO WITH ENDING")
    print("="*60)
    
    # Configuration for audio combination
    SILENT_GAP_MS = 1000   # Milliseconds of silence between narration and ending (0-3000)
    CROSSFADE_MS = 0       # Crossfade duration in milliseconds (0 = no overlap, 200-500 = smooth)
    
    # Validate ending file
    import os
    if not os.path.exists(ENDING_PATH):
        print(f"❌ ERROR: Ending file not found: {ENDING_PATH}")
        ENDING_ENABLED = False
    else:
        file_size = os.path.getsize(ENDING_PATH)
        print(f"ℹ️  Ending file: {ENDING_PATH} ({file_size} bytes)")
        
        # Check if file is valid
        if file_size == 0:
            print(f"❌ ERROR: Ending file is empty!")
            ENDING_ENABLED = False

if ENDING_ENABLED:
    # Load the generated narration
    narration = AudioSegment.from_wav("/content/mystery_narration.wav")
    print(f"✓ Loaded narration: {len(narration) / 1000:.1f} seconds")
    
    # Load the ending snippet - use from_file() for auto-detection
    print(f"Loading ending from: {ENDING_PATH}")
    try:
        # Use from_file() instead of from_mp3() - it auto-detects format
        ending = AudioSegment.from_file(ENDING_PATH)
        print(f"✓ Loaded ending: {len(ending) / 1000:.1f} seconds")
    except Exception as e:
        print(f"❌ ERROR: Failed to load ending file!")
        print(f"   Error: {str(e)}")
        print(f"\n   Troubleshooting:")
        print(f"   1. Make sure the file is a valid audio file (MP3, WAV, M4A, etc.)")
        print(f"   2. Try re-uploading the file")
        print(f"   3. Test the file on your computer first")
        print(f"\n   Skipping ending combination...")
        ENDING_ENABLED = False

if ENDING_ENABLED:
    # Combine audio with silent gap and optional crossfade
    print(f"\nCombining audio:")
    print(f"  - Silent gap: {SILENT_GAP_MS}ms ({SILENT_GAP_MS / 1000:.1f}s)")
    print(f"  - Crossfade: {CROSSFADE_MS}ms ({CROSSFADE_MS / 1000:.1f}s)")
    
    if SILENT_GAP_MS > 0:
        # Add silent gap between narration and ending
        silent_gap = AudioSegment.silent(duration=SILENT_GAP_MS)
        combined = narration + silent_gap
    else:
        combined = narration
    
    if CROSSFADE_MS > 0:
        # Add ending with crossfade (smooth transition)
        combined = combined.append(ending, crossfade=CROSSFADE_MS)
        print(f"✓ Applied {CROSSFADE_MS}ms crossfade for smooth transition")
    else:
        # Add ending without crossfade (no overlap)
        combined = combined + ending
        print(f"✓ No crossfade - clean separation")
    
    total_duration = len(combined) / 1000 / 60
    print(f"✓ Combined audio: {total_duration:.2f} minutes")
    
    # Export final combined audio
    final_output_path = "/content/mystery_podcast_final.mp3"
    combined.export(final_output_path, format="mp3", bitrate="192k")
    print(f"✓ Saved final podcast to: {final_output_path}")
    
    print("\n🎧 Listen to the final podcast with ending:")
    display(Audio(final_output_path))

if not ENDING_ENABLED:
    print("\nℹ️  Skipping ending combination (no valid ending file)")

## Step 7: Download Your Audio

Download the generated audio to your laptop:

In [ ]:
from google.colab import files

# Download the appropriate file
if ENDING_ENABLED:
    # Download the final combined podcast
    files.download('/content/mystery_podcast_final.mp3')
    print("✓ Downloaded: mystery_podcast_final.mp3 (with ending)")
else:
    # Download just the narration
    files.download('/content/mystery_narration.wav')
    print("✓ Downloaded: mystery_narration.wav (narration only)")

print("\n✓ Download started! Check your browser's download folder.")
print("\nNext steps:")
print("1. Listen to the audio and evaluate quality")
print("2. If satisfied, you can use this workflow for future stories")
print("3. Adjust CFG_SCALE (1.0-2.0) or INFERENCE_STEPS (5-20) for different quality")
print("4. Upload an ending.mp3 file to add professional closing to your podcast")

## Tips for Best Results

### Text Formatting Requirements ⚠️

**CRITICAL**: VibeVoice community fork requires `Speaker N:` format!

**Correct format for single-narrator:**
```
Speaker 0: It was a dark and stormy night. The fog rolled in from the harbor.

Speaker 0: Inside the mansion, something stirred in the shadows.

Speaker 0: The detective knew this case would be unlike any other.
```

**Correct format for multi-speaker dialogue:**
```
Speaker 0: The detective entered the study.
Speaker 1: What do you know about the missing artifact?
Speaker 2: Nothing! I've told you everything!
Speaker 0: But the detective noticed something odd in the corner.
```

**INCORRECT formats that cause errors:**
```
[speaker 0] Your text...                  ❌ Wrong - uses brackets
NARRATOR: Your text...                    ❌ Wrong - wrong keyword
Alice: Your text...                       ❌ Wrong - must use "Speaker"
It was a dark and stormy night...         ❌ Wrong - no label
Speaker0: Text...                         ❌ Wrong - missing space after "Speaker"
Speaker 0 Text...                         ❌ Wrong - missing colon
```

**Format requirements:**
- **Keyword**: Must start with `Speaker` (case-insensitive)
- **Space**: Required after "Speaker" → `Speaker 0:` not `Speaker0:`
- **Number**: Use 0 for single narrator, 0-3 for multiple speakers
- **Colon**: Required after number → `Speaker 0:` not `Speaker 0`
- **Text**: Follows the colon with a space

**Valid variations (all work):**
- `Speaker 0: Your text...` ✓
- `speaker 0: Your text...` ✓
- `SPEAKER 0: Your text...` ✓
- `Speaker  0  :  Your text...` ✓ (extra spaces OK)

### Ending Snippet Configuration (Optional)

Add a professional closing to your mystery podcast with customizable gap and crossfade:

**Step 6.5 Cell 15 Configuration:**
```python
SILENT_GAP_MS = 1000   # Silent pause between narration and ending (milliseconds)
CROSSFADE_MS = 0       # Crossfade overlap duration (0 = no overlap)
```

**Silent Gap Options:**
- `0ms` - No gap (immediate transition)
- `500ms` - Quick pause (0.5 seconds)
- `1000ms` - Standard pause (1 second) ⭐ Recommended
- `2000ms` - Long pause (2 seconds)
- `3000ms` - Extended pause (3 seconds)

**Crossfade Options:**
- `0ms` - No crossfade, clean separation ⭐ Recommended (prevents overlap)
- `200ms` - Subtle blend (0.2 seconds)
- `500ms` - Smooth transition (0.5 seconds)
- `1000ms` - Gradual fade (1 second, may overlap voices)

**Recommended Combinations:**

1. **Clean Separation** (no overlap):
   ```python
   SILENT_GAP_MS = 1000
   CROSSFADE_MS = 0
   ```
   Best for: Narration ending with music/credits starting

2. **Smooth Transition** (gentle blend):
   ```python
   SILENT_GAP_MS = 500
   CROSSFADE_MS = 300
   ```
   Best for: Background music fading in during final words

3. **Direct Connection** (no gap):
   ```python
   SILENT_GAP_MS = 0
   CROSSFADE_MS = 0
   ```
   Best for: Continuous audio flow

**Upload Instructions:**
- **Upload ending file**: Click folder icon → upload ending.mp3 or ending.wav
- **Set path**: Update `ENDING_PATH = "/content/ending.mp3"` in Cell 14
- **Format**: MP3, WAV, M4A, or other common audio formats
- **Output**: Final MP3 at 192kbps bitrate
- **Use case**: Theme music, credits, "subscribe" calls-to-action

### Voice Reference Quality

- **Length**: 5-15 seconds ideal
- **Quality**: Clean audio, no background noise
- **Content**: Natural speaking, not shouting/whispering
- **Male voice**: Works best with male narrator samples

### Generation Parameters

- **CFG Scale**:
  - 1.0 = More creative, natural variation
  - 1.3 = Balanced (recommended)
  - 2.0 = Stricter text adherence, less variation

- **Inference Steps**:
  - 5 steps = Fastest (2-3x faster than 20 steps)
  - 10 steps = Good balance
  - 20 steps = Best quality (slower)

### Text Preparation

- **Formatting**: Clean paragraphs, proper punctuation, **`Speaker N:` labels required**
- **Length**: Can handle up to ~50,000 words
- **Pacing**: Model automatically adds dramatic pauses
- **Emotion**: Suspenseful tone emerges naturally from content

---

## Cost

- **Colab Free Tier**: Completely free! (limited GPU hours per month)
- **Colab Pro ($10/month)**: More GPU hours, faster GPUs (A100)

**Typical usage**:
- 10-minute mystery: ~60 min GPU time
- 30-minute podcast: ~3 hours GPU time
- Free tier gives ~12 hours/week (enough for 2-3 episodes)

---

## Troubleshooting

**"No valid speaker lines found in script"**
- **CAUSE**: Missing or incorrect speaker labels
- **FIX**: Use `Speaker 0:` format (with colon!)
- **Example**: `Speaker 0: Your text here...`
- **Note**: Must use the word "Speaker", not "NARRATOR" or character names

**"Could not parse line" warnings**
- **CAUSE**: Lines without proper `Speaker N:` format
- **FIX**: Ensure ALL text lines start with `Speaker 0:` or `Speaker 1:`, etc.
- **Check**: Space after "Speaker" and colon after number
- **Common mistake**: `[speaker 0]` (brackets) instead of `Speaker 0:` (colon)

**"Narration and ending overlap"**
- **CAUSE**: `CROSSFADE_MS` set too high (causing audio overlap)
- **FIX**: Set `CROSSFADE_MS = 0` for clean separation
- **Alternative**: Increase `SILENT_GAP_MS` to add more pause before ending

**"CouldntDecodeError: ffmpeg returned error code 1"**
- **CAUSE**: Corrupted or invalid audio file
- **FIX**: Re-upload the ending file, ensure it's a valid audio format
- **Test**: Play the file on your computer before uploading

**"CUDA out of memory"**
- Reduce INFERENCE_STEPS to 5
- Split very long stories into chapters
- Request different GPU type (Runtime > Change runtime)

**"Generation too slow"**
- Reduce INFERENCE_STEPS from 20 → 5 (2-3x faster)
- Use Colab Pro for A100 GPU (2x faster than T4)

**"Audio quality poor"**
- Increase INFERENCE_STEPS to 10-20
- Use higher quality voice reference
- Increase CFG_SCALE to 1.5-2.0

---

## Next Steps

After testing VibeVoice here:
1. If quality is excellent → Consider cloud GPU for production (RunPod/Vast.ai)
2. If quality is just OK → Stick with local XTTS/F5-TTS
3. Compare to your current engines (XTTS, Chatterbox, F5-TTS)

See `/docs/CLOUD_GPU_VIBEVOICE_SETUP.md` for production cloud GPU setup.

---

**Notebook created for**: Antique Mystery Podcast Generator  
**VibeVoice**: Microsoft Research (MIT License)  
**Community Fork**: https://github.com/vibevoice-community/VibeVoice